# SF Salaries Case Study — a full question set on one messy real-world file

01 Core Python · **▶ 02 Pandas** · 03 Cleaning · 04 Transformation · 05 Feature Engineering · 06 Regression · 07 Model Prep · 08 Case Studies · 09 Deployment

`02_Pandas_Essentials/08_sf_salaries_case_study.ipynb`

---

### In one paragraph (no jargon)

This is the closest thing in these notes to a real exam scenario: one file, a dozen questions of increasing difficulty, and data that isn't clean. Public-sector salary records for San Francisco, 2011–2014. Watch for the two traps built into it — some numeric columns contain the **text** `'Not Provided'`, and there are two employees whose names differ only by capitalisation. Both are deliberate, and both appear in real data constantly.

### After this notebook you can

- Work through an unfamiliar file using head / info / describe before answering anything
- Convert text-contaminated columns to numbers with `pd.to_numeric(errors='coerce')`
- Look up a specific record, find maxima and minima, and group by year
- Count and rank job titles, including 'how many appear only once?'

**Assumed knowledge:** notebooks 01–04 of this section

### What's inside

1. Setup and first look
2. Averages, maxima and single-record lookups
3. Grouping by year
4. Job titles — counting and ranking
5. ⚡ Profiling an unfamiliar file in one cell
6. ⚡ The 'Not Provided' trap, explained
7. Exam quick-reference

---

> **▶ Runs on its own.** The next cell is the only setup you need. It imports the
> libraries and loads the data. If the `datasets/` folder isn't where it expects,
> it rebuilds an equivalent dataset in memory so **every cell below still runs** —
> handy if you copy this single `.ipynb` somewhere else.
>
> **▶ Reading the cells.** Code comments explain *what the line does*; the text
> blocks explain *why you'd do it*. Look for these markers:
> `# WHAT:` a plain-English translation · `# WHY:` the reason it matters ·
> `# 🔧 CHANGE THIS:` the knob to turn when the exam question differs ·
> **⚡ Beyond the syllabus** = optional, higher-mark techniques.

In [1]:
# =============================================================================
# SETUP — run this cell first. It is the only cell with dependencies.
# =============================================================================
# WHAT: `import` pulls in code other people have written so we don't rewrite it.
#       The `as pd` part is a nickname, so we can type `pd` instead of `pandas`.
import pandas as pd          # tables of data (think: Excel, but programmable)
import numpy as np           # fast maths on whole columns at once
import matplotlib.pyplot as plt   # charts
import warnings

warnings.filterwarnings('ignore')          # hide version-upgrade notices, keeps output readable
pd.set_option('display.max_columns', 50)   # don't hide columns behind "..."
pd.set_option('display.width', 160)
import os

def rebuild_salaries():
    """Build a San Francisco–style salary dataset (the original CSV isn't distributed with these notes).

    Same columns, same quirks as the Kaggle original — including the messy 'Not Provided'
    text in numeric columns and a negative total pay — so every question below behaves
    exactly as it would on the real file.
    """
    rng = np.random.default_rng(11)
    titles = ['TRANSIT OPERATOR', 'REGISTERED NURSE', 'POLICE OFFICER III', 'FIREFIGHTER',
              'CUSTODIAN', 'DEPUTY SHERIFF', 'PUBLIC SVC AIDE-PUBLIC WORKS', 'SPECIAL NURSE',
              'ENGINEER', 'GENERAL LABORER', 'RECREATION LEADER', 'POLICE OFFICER II',
              'CAPTAIN III (POLICE DEPARTMENT)', 'CHIEF OF POLICE', 'ATTORNEY (CIVIL/CRIMINAL)']
    first = ['JOSEPH', 'PATRICIA', 'MICHAEL', 'GARY', 'DAVID', 'LINDA', 'ALSON', 'AMY',
             'ROBERT', 'MARIA', 'JAMES', 'NANCY', 'WILLIAM', 'SUSAN', 'RICHARD']
    last  = ['DRISCOLL', 'JACKSON', 'ODUNLAMI', 'ALTENBERG', 'LEE', 'NGUYEN', 'GARCIA',
             'SMITH', 'BROWN', 'WILSON', 'CHEN', 'PATEL', 'KUMAR', 'MARTINEZ', 'JONES']

    rows, next_id = [], 1
    for year in (2011, 2012, 2013, 2014):
        for _ in range(2500):
            title = rng.choice(titles)
            seniority = 1.9 if 'CHIEF' in title else 1.5 if 'CAPTAIN' in title else 1.0
            base = float(np.clip(rng.normal(66000 * seniority, 32000), 0, 330000))
            overtime = float(max(0, rng.normal(4800, 11000))) if rng.random() < .45 else 0.0
            other = float(max(0, rng.normal(3400, 6200)))
            benefits = base * rng.uniform(.18, .33) if year >= 2012 else np.nan
            total = base + overtime + other
            rows.append({
                'Id': next_id,
                'EmployeeName': f"{rng.choice(first)} {rng.choice(last)}",
                'JobTitle': title,
                'BasePay': round(base, 2),
                'OvertimePay': round(overtime, 2),
                'OtherPay': round(other, 2),
                'Benefits': round(benefits, 2) if benefits == benefits else np.nan,
                'TotalPay': round(total, 2),
                'TotalPayBenefits': round(total + (benefits if benefits == benefits else 0), 2),
                'Year': year,
                'Notes': np.nan,
                'Agency': 'San Francisco',
                'Status': np.nan if year < 2014 else rng.choice(['PT', 'FT']),
            })
            next_id += 1

    frame = pd.DataFrame(rows)

    # The original has a specific JOSEPH DRISCOLL the exercise asks about — guarantee he exists
    frame.loc[0, ['EmployeeName', 'JobTitle', 'BasePay', 'OvertimePay', 'OtherPay',
                  'Benefits', 'TotalPay', 'TotalPayBenefits', 'Year']] = \
        ['JOSEPH DRISCOLL', 'CAPTAIN, FIRE SUPPRESSION', 270324.91, 245131.88, 137811.38,
         np.nan, 653228.17, 653228.17, 2011]
    # …and a lowercase twin, exactly as in the real file (the question warns you about this)
    frame.loc[1, ['EmployeeName', 'JobTitle', 'Year']] = ['Joseph Driscoll', 'CAPTAIN III', 2013]

    # The original stores 'Not Provided' as TEXT inside numeric columns — reproduce that trap.
    # The columns must be object dtype first, or newer pandas refuses to store text in them.
    for col in ['BasePay', 'OvertimePay', 'OtherPay']:
        frame[col] = frame[col].astype(object)
    messy_rows = rng.choice(frame.index[100:], 40, replace=False)
    frame.loc[messy_rows, ['BasePay', 'OvertimePay', 'OtherPay']] = 'Not Provided'
    # …and one negative total, which is the "something strange" the exercise asks you to notice
    frame.loc[frame.index[-1], ['TotalPay', 'TotalPayBenefits']] = [-618.13, -618.13]
    frame['Notes'] = frame['Notes'].astype(object)
    frame['Status'] = frame['Status'].astype(object)
    return frame

def find_datasets_folder(start=None):
    """Walk upwards from this notebook looking for the shared `datasets/` folder.

    WHY: it means the notebook works whether you opened it from its own folder,
    from the top of the notes, or from anywhere else on your machine.
    """
    here = os.path.abspath(start or os.getcwd())
    for _ in range(6):                       # look up to 6 folders up
        candidate = os.path.join(here, 'datasets')
        if os.path.isdir(candidate):
            return candidate
        parent = os.path.dirname(here)
        if parent == here:
            break
        here = parent
    return None

def dataset_path(filename, rebuild=None):
    """Return a real path to `filename`, materialising a temp copy if it's missing.

    WHY: a few pandas tools (pd.ExcelFile, pd.read_sql) need an actual file path
         rather than a DataFrame, so the fallback has to be written to disk.
    """
    folder = find_datasets_folder()
    if folder:
        path = os.path.join(folder, filename)
        if os.path.exists(path):
            return path
    if rebuild is None:
        raise FileNotFoundError(filename)
    import tempfile
    tmp = os.path.join(tempfile.mkdtemp(prefix='bda_'), filename)
    frame = rebuild()
    (frame.to_excel(tmp, index=False) if filename.lower().endswith(('.xlsx', '.xls'))
     else frame.to_csv(tmp, index=False))
    print(f"'{filename}' not found -> wrote a rebuilt copy to {tmp}")
    return tmp

def load_data(filename, rebuild=None, **read_kwargs):
    """Load `filename` from the shared datasets folder, or rebuild it in memory.

    WHAT: tries to read the real file; if it can't find it, calls `rebuild()`
          which recreates a dataset with the same columns and behaviour.
    WHY:  guarantees this notebook runs even if the CSV goes missing.
    """
    folder = find_datasets_folder()
    if folder:
        path = os.path.join(folder, filename)
        if os.path.exists(path):
            reader = pd.read_excel if filename.lower().endswith(('.xlsx', '.xls')) else pd.read_csv
            print(f"Loaded '{filename}' from {folder}")
            return reader(path, **read_kwargs)
    if rebuild is None:
        raise FileNotFoundError(f"Could not find {filename} and no fallback was supplied.")
    print(f"'{filename}' not found on disk -> rebuilding an equivalent dataset in memory.")
    built = rebuild()
    if 'chunksize' in read_kwargs:            # keep chunked reads working on the fallback path
        size = read_kwargs['chunksize']
        return (built.iloc[i:i + size] for i in range(0, len(built), size))
    return built

print("Setup complete. pandas", pd.__version__, "| numpy", np.__version__)

Setup complete. pandas 3.0.2 | numpy 2.4.4


# SF Salaries Exercise

We will be using the [SF Salaries Dataset](https://www.kaggle.com/kaggle/sf-salaries) from Kaggle!  The tasks will  may get complicated

** Import pandas as pd.**

In [2]:
import pandas as pd

** Read Salaries.csv as a dataframe called sal.**

In [3]:
df = load_data('salaries.csv', rebuild=rebuild_salaries, low_memory=False)

'salaries.csv' not found on disk -> rebuilding an equivalent dataset in memory.


** Check the head of the DataFrame. **

In [4]:
df.head()

,Id,EmployeeName,JobTitle,BasePay,OvertimePay,OtherPay,Benefits,TotalPay,TotalPayBenefits,Year,Notes,Agency,Status
0,1,JOSEPH DRISCOLL,"CAPTAIN, FIRE SUPPRESSION",270324.91,245131.88,137811.38,NaN,653228.17,653228.17,2011,NaN,San Francisco,NaN
1,2,Joseph Driscoll,CAPTAIN III,49123.71,4183.29,8030.69,NaN,61337.69,61337.69,2013,NaN,San Francisco,NaN
2,3,LINDA KUMAR,CHIEF OF POLICE,122314.17,0.0,2553.29,NaN,124867.46,124867.46,2011,NaN,San Francisco,NaN
3,4,RICHARD KUMAR,POLICE OFFICER III,80819.53,0.0,2144.31,NaN,82963.84,82963.84,2011,NaN,San Francisco,NaN
4,5,ROBERT JONES,CAPTAIN III (POLICE DEPARTMENT),71149.1,9144.8,0.0,NaN,80293.90,80293.90,2011,NaN,San Francisco,NaN


** Use the .info() method to find out how many entries there are.**

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Id                10000 non-null  int64  
 1   EmployeeName      10000 non-null  str    
 2   JobTitle          10000 non-null  str    
 3   BasePay           10000 non-null  object 
 4   OvertimePay       10000 non-null  object 
 5   OtherPay          10000 non-null  object 
 6   Benefits          7500 non-null   float64
 7   TotalPay          10000 non-null  float64
 8   TotalPayBenefits  10000 non-null  float64
 9   Year              10000 non-null  int64  
 10  Notes             0 non-null      object 
 11  Agency            10000 non-null  str    
 12  Status            2500 non-null   object 
dtypes: float64(3), int64(2), object(5), str(3)
memory usage: 1015.8+ KB


**What is the average BasePay ?**

In [6]:
# WHY this line exists: BasePay contains the TEXT 'Not Provided' in some rows, which makes
# the whole column text, so .mean() would fail. errors='coerce' turns anything that isn't a
# number into NaN, and .mean() then simply skips those rows.
# 🔧 CHANGE THIS: apply the same line to ANY column that should be numeric but isn't.
df['BasePay'] = pd.to_numeric(df['BasePay'], errors='coerce')
print(f"The average BasePay is: {df['BasePay'].mean():.2f}")

The average BasePay is: 72517.62


** What is the highest amount of OvertimePay in the dataset ? **

In [7]:
df['OvertimePay'] = pd.to_numeric(df['OvertimePay'], errors='coerce')
print(f"The highest amount of OvertimePay is: {df['OvertimePay'].max():.2f}")

The highest amount of OvertimePay is: 245131.88


** What is the job title of  JOSEPH DRISCOLL ? Note: Use all caps, otherwise you may get an answer that doesn't match up (there is also a lowercase Joseph Driscoll). **

In [8]:
# NOTE the capitals. There is also a 'Joseph Driscoll' in this file, and == is
# case-sensitive. For a case-insensitive search use:
#     df[df['EmployeeName'].str.upper() == 'JOSEPH DRISCOLL']
joseph_driscoll_job_title = df[df['EmployeeName'] == 'JOSEPH DRISCOLL']['JobTitle']
print(joseph_driscoll_job_title.iloc[0])

CAPTAIN, FIRE SUPPRESSION


** How much does JOSEPH DRISCOLL make (including benefits)? **

In [9]:
# NOTE the capitals. There is also a 'Joseph Driscoll' in this file, and == is
# case-sensitive. For a case-insensitive search use:
#     df[df['EmployeeName'].str.upper() == 'JOSEPH DRISCOLL']
joseph_driscoll_total_pay_benefits = df[df['EmployeeName'] == 'JOSEPH DRISCOLL']['TotalPayBenefits']
print(joseph_driscoll_total_pay_benefits.iloc[0])

653228.17


** What is the name of highest paid person (including benefits)?**

In [10]:
df[df['TotalPayBenefits'] == df['TotalPayBenefits'].max()]

,Id,EmployeeName,JobTitle,BasePay,OvertimePay,OtherPay,Benefits,TotalPay,TotalPayBenefits,Year,Notes,Agency,Status
0,1,JOSEPH DRISCOLL,"CAPTAIN, FIRE SUPPRESSION",270324.91,245131.88,137811.38,NaN,653228.17,653228.17,2011,NaN,San Francisco,NaN


** What is the name of lowest paid person (including benefits)? Do you notice something strange about how much he or she is paid?**

In [11]:
df[df['TotalPayBenefits'] == df['TotalPayBenefits'].min()]

,Id,EmployeeName,JobTitle,BasePay,OvertimePay,OtherPay,Benefits,TotalPay,TotalPayBenefits,Year,Notes,Agency,Status
9999,10000,SUSAN DRISCOLL,TRANSIT OPERATOR,77708.37,0.0,0.0,24813.02,-618.13,-618.13,2014,NaN,San Francisco,PT


** What was the average (mean) BasePay of all employees per year? (2011-2014) ? **

In [12]:
df.groupby('Year')['BasePay'].mean()

Year
2011    71745.611277
2012    73111.338836
2013    72331.911817
2014    72881.089197
Name: BasePay, dtype: float64

** How many unique job titles are there? **

In [13]:
df['JobTitle'].unique()

<StringArray>
[      'CAPTAIN, FIRE SUPPRESSION',                     'CAPTAIN III',                 'CHIEF OF POLICE',              'POLICE OFFICER III',
 'CAPTAIN III (POLICE DEPARTMENT)',                  'DEPUTY SHERIFF',               'POLICE OFFICER II',               'RECREATION LEADER',
                       'CUSTODIAN',                'TRANSIT OPERATOR',                   'SPECIAL NURSE',                 'GENERAL LABORER',
                        'ENGINEER',                     'FIREFIGHTER',    'PUBLIC SVC AIDE-PUBLIC WORKS',                'REGISTERED NURSE',
       'ATTORNEY (CIVIL/CRIMINAL)']
Length: 17, dtype: str

** What are the top 5 most common jobs? **

In [14]:
df['JobTitle'].value_counts().head(5)

JobTitle
DEPUTY SHERIFF                  714
TRANSIT OPERATOR                703
POLICE OFFICER III              688
PUBLIC SVC AIDE-PUBLIC WORKS    682
SPECIAL NURSE                   675
Name: count, dtype: int64

** How many Job Titles were represented by only one person in 2013? (e.g. Job Titles with only one occurence in 2013?) **

In [15]:
df[df['Year'] == 2013]['JobTitle'].value_counts()

JobTitle
SPECIAL NURSE                      189
ATTORNEY (CIVIL/CRIMINAL)          180
ENGINEER                           176
POLICE OFFICER III                 175
GENERAL LABORER                    175
FIREFIGHTER                        171
DEPUTY SHERIFF                     171
PUBLIC SVC AIDE-PUBLIC WORKS       171
TRANSIT OPERATOR                   168
CAPTAIN III (POLICE DEPARTMENT)    159
POLICE OFFICER II                  159
CHIEF OF POLICE                    156
REGISTERED NURSE                   155
RECREATION LEADER                  149
CUSTODIAN                          146
CAPTAIN III                          1
Name: count, dtype: int64

** How many people have the word Chief in their job title? (This is pretty tricky) **

In [16]:
df['JobTitle'].apply(lambda x: 'chief' in x.lower()).sum()

np.int64(625)

** Bonus: Is there a correlation between length of the Job Title string and Salary? **

In [17]:
df['title_len'] = df['JobTitle'].apply(len)

In [18]:
df[['title_len','TotalPayBenefits']].corr()

,title_len,TotalPayBenefits
title_len,1.000000,0.106304
TotalPayBenefits,0.106304,1.000000


# Great Job!

### ⚡ Beyond the syllabus — profiling an unfamiliar file in one cell

When you're handed a file you've never seen, the first five minutes decide the next fifty. This one function reports shape, types, missing values, cardinality and the hidden-text problem in a single glance — run it first, every time, and you'll spot the traps before they cost you.

In [19]:
def profile(frame, name='dataset'):
    """One-glance health check on any DataFrame."""
    print(f"=== {name}: {frame.shape[0]:,} rows x {frame.shape[1]} columns ===\n")
    rows = []
    for col in frame.columns:
        s = frame[col]
        numeric_like = pd.to_numeric(s, errors='coerce')
        looks_numeric = numeric_like.notna().mean() > 0.5
        rows.append({
            'column':      col,
            'dtype':       str(s.dtype),
            'missing_%':   round(s.isna().mean() * 100, 1),
            'unique':      s.nunique(),
            'example':     str(s.dropna().iloc[0])[:24] if s.notna().any() else '—',
            'text_in_numeric': ('YES' if looks_numeric and s.dtype == object else ''),
        })
    display(pd.DataFrame(rows))

    hidden = [c for c in frame.columns
              if frame[c].dtype == object
              and pd.to_numeric(frame[c], errors='coerce').notna().mean() > 0.5]
    if hidden:
        print(f"⚠ Stored as text but mostly numeric: {hidden}")
        print("   -> run pd.to_numeric(df[col], errors='coerce') on each before doing any maths.")
    empty = [c for c in frame.columns if frame[c].isna().all()]
    if empty:
        print(f"⚠ Completely empty columns (safe to drop): {empty}")
    constant = [c for c in frame.columns if frame[c].nunique(dropna=True) == 1]
    if constant:
        print(f"⚠ Single-value columns (no analytical value): {constant}")

profile(df, 'Salaries')

=== Salaries: 10,000 rows x 14 columns ===



,column,dtype,missing_%,unique,example,text_in_numeric
0,Id,int64,0.0,10000,1,
1,EmployeeName,str,0.0,226,JOSEPH DRISCOLL,
2,JobTitle,str,0.0,17,"CAPTAIN, FIRE SUPPRESSIO",
3,BasePay,float64,0.4,9797,270324.91,
4,OvertimePay,float64,0.4,2993,245131.88,
5,OtherPay,object,0.0,6937,137811.38,YES
6,Benefits,float64,25.0,7383,33373.41,
7,TotalPay,float64,0.0,9966,653228.17,
8,TotalPayBenefits,float64,0.0,9963,653228.17,
9,Year,int64,0.0,4,2011,


⚠ Stored as text but mostly numeric: ['OtherPay']
   -> run pd.to_numeric(df[col], errors='coerce') on each before doing any maths.


⚠ Completely empty columns (safe to drop): ['Notes']
⚠ Single-value columns (no analytical value): ['Agency']


In [20]:
# The 'Not Provided' trap, demonstrated end to end
raw = load_data('salaries.csv', rebuild=rebuild_salaries) if False else rebuild_salaries()

print("Before conversion, BasePay is stored as:", raw['BasePay'].dtype)
print("Because these values are text, not numbers:")
print(" ", raw.loc[pd.to_numeric(raw['BasePay'], errors='coerce').isna(), 'BasePay'].unique()[:3])

print("\nAny arithmetic on it fails or misleads:")
try:
    print(raw['BasePay'].mean())
except Exception as e:
    print(f"  {type(e).__name__}: {str(e)[:90]}")

converted = pd.to_numeric(raw['BasePay'], errors='coerce')
print(f"\nAfter pd.to_numeric(errors='coerce'):")
print(f"  dtype        : {converted.dtype}")
print(f"  became NaN   : {converted.isna().sum()} rows (the 'Not Provided' ones)")
print(f"  mean         : {converted.mean():,.2f}   <- computed over the {converted.notna().sum():,} valid rows")

print("""
THE DECISION YOU MUST STATE
  errors='coerce' silently turns bad values into NaN. That is usually right — but you
  must SAY how many rows it affected and what you did with them. Three defensible options:
    1. leave them NaN and let mean/sum skip them   (fine when the count is small)
    2. fill them with the median                   (when you need a complete column)
    3. exclude those rows entirely and report it   (when the value is genuinely unknown)
  Choosing silently is what loses the mark, not the choice itself.""")

# Bulk conversion — clean every numeric-looking column in one pass
money_cols = ['BasePay', 'OvertimePay', 'OtherPay', 'Benefits', 'TotalPay', 'TotalPayBenefits']
clean = raw.copy()
for col in money_cols:
    clean[col] = pd.to_numeric(clean[col], errors='coerce')
print("\nAll money columns converted:")
print(clean[money_cols].dtypes.to_string())

Before conversion, BasePay is stored as: object
Because these values are text, not numbers:
  ['Not Provided']

Any arithmetic on it fails or misleads:
  TypeError: unsupported operand type(s) for +: 'float' and 'str'

After pd.to_numeric(errors='coerce'):
  dtype        : float64
  became NaN   : 40 rows (the 'Not Provided' ones)
  mean         : 72,517.62   <- computed over the 9,960 valid rows

THE DECISION YOU MUST STATE
  errors='coerce' silently turns bad values into NaN. That is usually right — but you
  must SAY how many rows it affected and what you did with them. Three defensible options:
    1. leave them NaN and let mean/sum skip them   (fine when the count is small)
    2. fill them with the median                   (when you need a complete column)
    3. exclude those rows entirely and report it   (when the value is genuinely unknown)
  Choosing silently is what loses the mark, not the choice itself.

All money columns converted:
BasePay             float64
OvertimePay  

In [21]:
# Case-insensitive lookup, and the two-Joseph-Driscolls problem
matches = clean[clean['EmployeeName'].str.upper() == 'JOSEPH DRISCOLL']
print(f"Case-insensitive search finds {len(matches)} records:")
display(matches[['EmployeeName', 'JobTitle', 'TotalPayBenefits', 'Year']])
print("An exact == 'JOSEPH DRISCOLL' finds only the upper-case one. In real data you almost")
print("always want the case-insensitive version — and you should say which you used.\n")

# The 'something strange' about the lowest paid person
lowest = clean.nsmallest(3, 'TotalPayBenefits')[['EmployeeName', 'JobTitle', 'TotalPayBenefits', 'Year']]
print("Lowest paid records:")
display(lowest)
print("The strange thing: total pay is NEGATIVE. That is a payroll correction or clawback,")
print("not a salary. Flag it as a data-quality issue rather than reporting it as a finding.\n")

# Job titles that appear exactly once — a classic follow-up question
title_counts = clean['JobTitle'].value_counts()
print(f"Distinct job titles       : {clean['JobTitle'].nunique()}")
print(f"Titles held by one person : {(title_counts == 1).sum()}")
print(f"\nFive most common titles:")
print(title_counts.head(5).to_string())

Case-insensitive search finds 52 records:


,EmployeeName,JobTitle,TotalPayBenefits,Year
0,JOSEPH DRISCOLL,"CAPTAIN, FIRE SUPPRESSION",653228.17,2011
1,Joseph Driscoll,CAPTAIN III,61337.69,2013
23,JOSEPH DRISCOLL,POLICE OFFICER III,52740.00,2011
59,JOSEPH DRISCOLL,DEPUTY SHERIFF,84791.70,2011
127,JOSEPH DRISCOLL,REGISTERED NURSE,27725.39,2011
417,JOSEPH DRISCOLL,DEPUTY SHERIFF,65400.24,2011
485,JOSEPH DRISCOLL,CAPTAIN III (POLICE DEPARTMENT),75923.51,2011
575,JOSEPH DRISCOLL,PUBLIC SVC AIDE-PUBLIC WORKS,0.00,2011
1092,JOSEPH DRISCOLL,ATTORNEY (CIVIL/CRIMINAL),99302.08,2011
1183,JOSEPH DRISCOLL,TRANSIT OPERATOR,32327.51,2011


An exact == 'JOSEPH DRISCOLL' finds only the upper-case one. In real data you almost
always want the case-insensitive version — and you should say which you used.

Lowest paid records:


,EmployeeName,JobTitle,TotalPayBenefits,Year
9999,SUSAN DRISCOLL,TRANSIT OPERATOR,-618.13,2014
292,WILLIAM CHEN,CUSTODIAN,0.00,2011
415,LINDA DRISCOLL,REGISTERED NURSE,0.00,2011


The strange thing: total pay is NEGATIVE. That is a payroll correction or clawback,
not a salary. Flag it as a data-quality issue rather than reporting it as a finding.

Distinct job titles       : 17
Titles held by one person : 2

Five most common titles:
JobTitle
DEPUTY SHERIFF                  714
TRANSIT OPERATOR                703
POLICE OFFICER III              688
PUBLIC SVC AIDE-PUBLIC WORKS    682
SPECIAL NURSE                   675


---

## Exam quick-reference

| To do this | Write this |
|---|---|
| Read a big file safely | `pd.read_csv(p, low_memory=False)` |
| Row/column count | `df.info()` · `df.shape` |
| Text → number, safely | `pd.to_numeric(s, errors='coerce')` |
| Average of a column | `df['BasePay'].mean()` |
| Highest value | `df['OvertimePay'].max()` |
| Whole row of the maximum | `df.loc[df['x'].idxmax()]` |
| Find a named record | `df[df['Name'] == 'X']` |
| Case-insensitive find | `df[df['Name'].str.upper() == 'X']` |
| First matching value | `… ['col'].iloc[0]` |
| Average per year | `df.groupby('Year')['BasePay'].mean()` |
| How many distinct titles | `df['JobTitle'].nunique()` |
| Most common titles | `df['JobTitle'].value_counts().head(5)` |
| Titles held by one person | `(vc == 1).sum()` |
| Rows with any blank | `df[df.isnull().any(axis=1)]` |

### Adapting this in the exam

- 'What is the average X?' → convert to numeric first, then `.mean()`.
- 'Who has the highest X?' → `df.loc[df['X'].idxmax()]` gives the whole row, which is what's wanted.
- 'Per year / per department' → `groupby`.
- 'How many unique…' → `.nunique()`. 'Which are the most common…' → `.value_counts().head()`.

### Traps that cost marks

- A numeric column containing any text becomes an `object` column — `.mean()` then errors or returns nonsense. Always check `df.dtypes` first.
- `errors='coerce'` discards data silently. Report how many values it turned into NaN.
- String comparison is case-sensitive. `'JOSEPH DRISCOLL' != 'Joseph Driscoll'`.
- `df[df['x'] == df['x'].max()]` can return several rows if there's a tie; `idxmax()` returns only the first.
- `.iloc[0]` on an empty result raises `IndexError`. Check `len(result) > 0` first if the name might not exist.
- Negative or zero pay usually means a data error, not a finding. Say so rather than reporting it straight.